# Exponential decay fit & half-life

Simulate a decay time course, fit a single exponential, and estimate the half-life with confidence bounds. One figure: data, fit and residuals.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy import stats

rng = np.random.default_rng(42)
t = np.linspace(0, 40, 24)
A0, k_true = 100.0, 0.12
meas = A0 * np.exp(-k_true * t) + rng.normal(0, 6, size=t.size)

def decay(t, A0, k): return A0 * np.exp(-k * t)

In [ ]:
popt, pcov = curve_fit(decay, t, meas, p0=(80, 0.1))
A0f, kf = popt
kerr = np.sqrt(np.diag(pcov))[1]
t_half = np.log(2) / kf
t_half_err = np.log(2) * kerr / kf**2
ci = stats.t.ppf(0.975, df=t.size - 2) * t_half_err
print(f'k      = {kf:.4f} ± {kerr:.4f} /h')
print(f't1/2   = {t_half:.2f} ± {t_half_err:.2f} h')
print(f'95% CI = [{t_half - ci:.2f}, {t_half + ci:.2f}] h')

In [ ]:
tfine = np.linspace(0, 40, 300)
fig, (ax, axr) = plt.subplots(2, 1, figsize=(7, 5), sharex=True,
                             gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.1})
ax.errorbar(t, meas, yerr=6, fmt='o', ms=4, color='#4f8cff', label='data')
ax.plot(tfine, decay(tfine, A0f, kf), color='#35c4b6', lw=2, label='fit')
ax.axvline(t_half, color='#d9a441', ls='--', label='t1/2')
ax.legend(); ax.set_ylabel('activity')
res = meas - decay(t, A0f, kf)
axr.scatter(t, res, s=12, color='#e05b5b')
axr.axhline(0, color='#8b97a5', lw=0.8); axr.set_xlabel('time (h)')